# 10b. A/B test simulation study

Notebook 10 ran **one** simulated trial. One trial cannot show whether the power calculation is right, because a single result can land anywhere.

This notebook repeats the simulated trial 2,000 times to answer two questions:

1. **Does the power calculation hold?** With 1,249 households per group, the calculation said 55% power to detect a 10% sales increase. Repeated trials should detect the effect in about 55% of runs.
2. **How much does baseline adjustment help?** Each household's sales in the previous 56 days predict its sales in the next 56 days. Adjusting for that baseline removes noise without changing the randomized comparison.

Everything here is a simulation. **No real randomized campaign was run.**

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import TTestIndPower

current_folder = Path.cwd()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder

transactions_path = project_root / "data" / "processed" / "transaction_data.parquet"
ab_report_folder = project_root / "reports" / "ab_test_design"

planning = pd.read_csv(ab_report_folder / "power_analysis.csv").iloc[0]
alpha = float(planning["alpha"])
planned_effect = float(planning["planned_absolute_difference"])
per_group = int(planning["available_per_group"])

print(f"Planned effect: {planned_effect:.2f}")
print(f"Households per group: {per_group:,}")
print(f"Calculated power from notebook 10: {planning['power_with_available_sample']:.2%}")

Planned effect: 27.43
Households per group: 1,249
Calculated power from notebook 10: 55.02%


## Planning cohort with a real baseline

Same cohort as notebook 10: households with a recorded transaction before day 531.

- **Outcome window:** days 531-586 (the planning window from notebook 10).
- **Baseline window:** days 475-530 (the 56 days before it).

Both windows are real recorded sales under business-as-usual marketing.

In [2]:
cohort = duckdb.sql(f"""
    WITH eligible AS (
        SELECT DISTINCT household_key
        FROM '{transactions_path}'
        WHERE DAY < 531
    )
    SELECT
        e.household_key,
        COALESCE(SUM(CASE WHEN t.DAY BETWEEN 475 AND 530 THEN t.SALES_VALUE END), 0) AS sales_baseline,
        COALESCE(SUM(CASE WHEN t.DAY BETWEEN 531 AND 586 THEN t.SALES_VALUE END), 0) AS sales_outcome
    FROM eligible AS e
    LEFT JOIN '{transactions_path}' AS t USING (household_key)
    GROUP BY e.household_key
    ORDER BY e.household_key
""").df()

assert cohort["household_key"].is_unique
assert len(cohort) == 2 * per_group

baseline_outcome_correlation = cohort["sales_baseline"].corr(cohort["sales_outcome"])

print(f"Households: {len(cohort):,}")
print(f"Mean outcome-window sales: {cohort['sales_outcome'].mean():.2f}")
print(f"Outcome standard deviation: {cohort['sales_outcome'].std(ddof=1):.2f}")
print(f"Correlation between baseline and outcome sales: {baseline_outcome_correlation:.3f}")

Households: 2,498
Mean outcome-window sales: 274.30
Outcome standard deviation: 328.46
Correlation between baseline and outcome sales: 0.855


## Part 1. Repeat the notebook 10 trial 2,000 times

Same recipe as notebook 10: draw each household's sales at random from the historical values, then add the planned effect to the campaign group. The only change is that we do it 2,000 times with fresh random draws.

In [3]:
n_trials = 2000
rng = np.random.default_rng(20261117)
outcome_values = cohort["sales_outcome"].to_numpy()


def welch_summary(campaign, holdout, alpha):
    """Welch t-test for many trials at once. Rows are trials."""
    n1, n0 = campaign.shape[1], holdout.shape[1]
    diff = campaign.mean(axis=1) - holdout.mean(axis=1)
    v1 = campaign.var(axis=1, ddof=1) / n1
    v0 = holdout.var(axis=1, ddof=1) / n0
    se = np.sqrt(v1 + v0)
    dof = (v1 + v0) ** 2 / (v1 ** 2 / (n1 - 1) + v0 ** 2 / (n0 - 1))
    p_value = 2 * stats.t.sf(np.abs(diff / se), dof)
    half_width = stats.t.ppf(1 - alpha / 2, dof) * se
    return diff, p_value, half_width


def resampled_trials(effect):
    draws = rng.choice(outcome_values, size=(n_trials, 2 * per_group), replace=True)
    campaign = draws[:, :per_group] + effect
    holdout = draws[:, per_group:]
    return welch_summary(campaign, holdout, alpha)


rows = []
for label, effect in [("Planned effect", planned_effect), ("No effect", 0.0)]:
    diff, p_value, half_width = resampled_trials(effect)
    rows.append({
        "design": "Resampled outcomes (notebook 10 recipe)",
        "analysis": "Welch t-test",
        "scenario": label,
        "true_effect": effect,
        "trials": n_trials,
        "mean_estimate": diff.mean(),
        "share_significant": (p_value < alpha).mean(),
        "interval_coverage": (np.abs(diff - effect) <= half_width).mean(),
        "mean_interval_half_width": half_width.mean(),
    })

part1 = pd.DataFrame(rows)
part1.round(4)

,design,analysis,scenario,true_effect,trials,mean_estimate,share_significant,interval_coverage,mean_interval_half_width
0,Resampled outcomes (notebook 10 recipe),Welch t-test,Planned effect,27.4296,2000,26.9102,0.5390,0.9520,25.7609
1,Resampled outcomes (notebook 10 recipe),Welch t-test,No effect,0.0000,2000,-0.3030,0.0615,0.9385,25.7362


**Reading the table**

- *Planned effect, share significant* is the simulated power. It should be close to the calculated 55%.
- *No effect, share significant* is the false-positive rate. It should be close to 5%.
- *Interval coverage* is how often the 95% interval contains the true effect. It should be close to 95%.

The single trial in notebook 10 (estimate 6.88, p = 0.60) was one of the roughly 45% of runs that miss an effect of this size.

## Part 2. Baseline adjustment with real household pairs

Part 1 drew outcomes at random, so a household's simulated sales had no link to its own history. Real households are consistent: heavy shoppers stay heavy shoppers.

Here each household keeps its **own real** baseline and outcome sales. Only the group assignment is random, and the planned effect is added to the campaign group.

Two analyses of the same trials:

- **Welch t-test** on outcome sales.
- **Baseline-adjusted** (CUPED): subtract the part of each household's outcome that its baseline already predicts, then run the same Welch test. The adjustment coefficient uses both groups pooled, so it does not use the group labels.

In [4]:
baseline = cohort["sales_baseline"].to_numpy()
no_campaign_outcome = cohort["sales_outcome"].to_numpy()
n_households = len(cohort)


def paired_trials(effect):
    results = {"Welch t-test": [], "Baseline-adjusted (CUPED)": []}
    for _ in range(n_trials):
        order = rng.permutation(n_households)
        in_campaign = np.zeros(n_households, dtype=bool)
        in_campaign[order[:per_group]] = True

        outcome = no_campaign_outcome + effect * in_campaign

        theta = np.cov(outcome, baseline, ddof=1)[0, 1] / baseline.var(ddof=1)
        adjusted = outcome - theta * (baseline - baseline.mean())

        for name, values in [("Welch t-test", outcome), ("Baseline-adjusted (CUPED)", adjusted)]:
            diff, p_value, half_width = welch_summary(
                values[in_campaign][None, :], values[~in_campaign][None, :], alpha
            )
            results[name].append((diff[0], p_value[0], half_width[0]))
    return {name: np.array(values) for name, values in results.items()}


rows = []
estimates_for_chart = {}
for label, effect in [("Planned effect", planned_effect), ("No effect", 0.0)]:
    for name, values in paired_trials(effect).items():
        diff, p_value, half_width = values[:, 0], values[:, 1], values[:, 2]
        if label == "Planned effect":
            estimates_for_chart[name] = diff
        rows.append({
            "design": "Real household pairs, random assignment",
            "analysis": name,
            "scenario": label,
            "true_effect": effect,
            "trials": n_trials,
            "mean_estimate": diff.mean(),
            "share_significant": (p_value < alpha).mean(),
            "interval_coverage": (np.abs(diff - effect) <= half_width).mean(),
            "mean_interval_half_width": half_width.mean(),
        })

part2 = pd.DataFrame(rows)
part2.round(4)

,design,analysis,scenario,true_effect,trials,mean_estimate,share_significant,interval_coverage,mean_interval_half_width
0,"Real household pairs, random assignment",Welch t-test,Planned effect,27.4296,2000,26.9791,0.5460,0.9500,25.7735
1,"Real household pairs, random assignment",Baseline-adjusted (CUPED),Planned effect,27.4296,2000,27.3233,0.9780,0.9510,13.3849
2,"Real household pairs, random assignment",Welch t-test,No effect,0.0000,2000,-0.2028,0.0440,0.9560,25.7735
3,"Real household pairs, random assignment",Baseline-adjusted (CUPED),No effect,0.0000,2000,-0.0187,0.0505,0.9495,13.3847


## What the adjustment means for sample size

The adjustment removes the share of outcome variance that the baseline explains. The remaining standard deviation is `sd x sqrt(1 - r^2)`, where `r` is the baseline-outcome correlation.

In [5]:
outcome_sd = float(planning["assumed_standard_deviation"])
adjusted_sd = outcome_sd * np.sqrt(1 - baseline_outcome_correlation ** 2)
power_model = TTestIndPower()

required_unadjusted = int(np.ceil(power_model.solve_power(
    effect_size=planned_effect / outcome_sd, alpha=alpha, power=0.80, ratio=1.0
)))
required_adjusted = int(np.ceil(power_model.solve_power(
    effect_size=planned_effect / adjusted_sd, alpha=alpha, power=0.80, ratio=1.0
)))
power_adjusted = power_model.power(
    effect_size=planned_effect / adjusted_sd, nobs1=per_group, alpha=alpha, ratio=1.0
)

sample_size_summary = pd.DataFrame([{
    "baseline_outcome_correlation": baseline_outcome_correlation,
    "outcome_sd": outcome_sd,
    "adjusted_sd": adjusted_sd,
    "required_per_group_unadjusted": required_unadjusted,
    "required_per_group_adjusted": required_adjusted,
    "available_per_group": per_group,
    "calculated_power_unadjusted": float(planning["power_with_available_sample"]),
    "calculated_power_adjusted": power_adjusted,
}])

sample_size_summary.T.round(4)

,0
baseline_outcome_correlation,0.8546
outcome_sd,328.4568
adjusted_sd,170.5763
required_per_group_unadjusted,2252.0000
required_per_group_adjusted,609.0000
available_per_group,1249.0000
calculated_power_unadjusted,0.5502
calculated_power_adjusted,0.9802


In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(-45, 100, 60)
ax.hist(estimates_for_chart["Welch t-test"], bins=bins, alpha=0.55, label="Welch t-test")
ax.hist(estimates_for_chart["Baseline-adjusted (CUPED)"], bins=bins, alpha=0.55,
        label="Baseline-adjusted (CUPED)")
ax.axvline(planned_effect, color="black", linestyle="--", linewidth=1)
ax.text(planned_effect + 1.5, ax.get_ylim()[1] * 0.93, f"True effect {planned_effect:.2f}")
ax.set_xlabel("Estimated difference in mean sales per household")
ax.set_ylabel("Simulated trials")
ax.set_title(f"{n_trials:,} simulated trials, {per_group:,} households per group")
ax.legend(frameon=False)
fig.tight_layout()

chart_path = ab_report_folder / "simulation_study_estimates.png"
fig.savefig(chart_path, dpi=150)
plt.close(fig)
print(f"Saved chart: {chart_path.relative_to(project_root)}")

Saved chart: reports/ab_test_design/simulation_study_estimates.png


In [7]:
simulation_study = pd.concat([part1, part2], ignore_index=True)
simulation_study.insert(0, "is_simulated", True)

simulation_study.to_csv(ab_report_folder / "simulation_study.csv", index=False)
sample_size_summary.to_csv(ab_report_folder / "baseline_adjustment_sample_size.csv", index=False)

print("Saved: reports/ab_test_design/simulation_study.csv")
print("Saved: reports/ab_test_design/baseline_adjustment_sample_size.csv")

Saved: reports/ab_test_design/simulation_study.csv
Saved: reports/ab_test_design/baseline_adjustment_sample_size.csv


## Limits

- These are simulations built on historical sales. They show how the design behaves, not what a real campaign would do.
- The imposed effect is the same for every campaign household.
- The baseline-outcome correlation comes from one pair of historical windows. A future period could be less predictable, which would shrink the gain.
- In a real experiment the baseline adjustment must be written into the analysis plan before the data is seen.